# BPE v09 Oluşturma Notebook'u

Bu notebook şunları yapar:
1. Hugging Face'ten 7 farklı veriset yükler (3 normal + 4 yorum)
2. Pandas DataFrame ile veri işleme
3. Yazım hatalarını yakalamak için yorumların ağırlığını artırır
4. BPE tokenleri oluşturur (ID aralığı: 22869-32767)
5. bpe_v09.json dosyasını oluşturur

## Verisetleri:
**Normal verisetler:**
- Wikipedia (wikimedia/wikipedia)
- Masallar (umutphp/masallar)
- Haberler (habanoz/news-tr-1.8M)

**Yorum verisetleri (yüksek ağırlık):**
- Hepsiburada yorumları 
- Beyazperde yorumları  
- Kitapyurdu yorumları
- Yorumbudur yorumları


In [3]:
# Gerekli kütüphaneleri import et
import json
import os
import re
from collections import Counter, defaultdict
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import pandas as pd
from tqdm import tqdm
import numpy as np
import random


## 1. Normal Verisetleri Yükle (Wikipedia, Masallar, Haberler)


In [4]:
# Wikipedia verisetini yükle
print("📚 Wikipedia verisetini yüklüyor...")
dswiki = load_dataset("wikimedia/wikipedia", "20231101.tr")
dfwiki = dswiki['train'].to_pandas()
print(f"✅ Wikipedia yüklendi. Boyut: {len(dfwiki):,} makale")
print(f"📊 Sütunlar: {dfwiki.columns.tolist()}")
dfwiki.head()


📚 Wikipedia verisetini yüklüyor...
✅ Wikipedia yüklendi. Boyut: 534,988 makale
📊 Sütunlar: ['id', 'url', 'title', 'text']


,id,url,title,text
0,10,https://tr.wikipedia.org/wiki/Cengiz%20Han,Cengiz Han,"Cengiz Han (doğum adıyla Temuçin, – 18 Ağusto..."
1,16,https://tr.wikipedia.org/wiki/Film%20%28anlam%...,Film (anlam ayrımı),Film şu anlamlara gelebilir:\n\n Camlara yapış...
2,22,https://tr.wikipedia.org/wiki/Mustafa%20Suphi,Mustafa Suphi,"Mehmed Mustafa Subhi (), kısaca Mustafa Suphi,..."
3,24,https://tr.wikipedia.org/wiki/Linux,Linux,Linux (telaffuz: Lin-uks); Linux çekirdeğine d...
4,30,https://tr.wikipedia.org/wiki/Bol%C5%9Fevizm,Bolşevizm,"Bolşevik, çoğunluktan yana anlamına gelen Rusç..."


In [5]:
# Masallar verisetini yükle
print("📖 Masallar verisetini yüklüyor...")
dsmasallar = load_dataset("umutphp/masallar")
dfmasallar = dsmasallar['train'].to_pandas()
print(f"✅ Masallar yüklendi. Boyut: {len(dfmasallar):,} masal")
print(f"📊 Sütunlar: {dfmasallar.columns.tolist()}")
dfmasallar.head()


📖 Masallar verisetini yüklüyor...
✅ Masallar yüklendi. Boyut: 1,528 masal
📊 Sütunlar: ['text', 'title', 'city', 'area']


,text,title,city,area
0,Eveli zamanda galbır zamanda bir yaşlı benim g...,Külcü Oğlan,Muğla,Ege Bölgesi
1,Bi adamın bi horuzu varmış horuzu. Horuzu almı...,[Mavi Yusuf],Muğla,Ege Bölgesi
2,Bir varmış bir yokmuş. Bir padişahın oğlu varm...,Padişah'ın Oğlu,Muğla,Ege Bölgesi
3,Evveli bi padişah varımış. O padişan iki tane ...,[Tuz Kadar],Muğla,Ege Bölgesi
4,Şimdi birisi varımış. Bu eskicilik yaparımış. ...,Azrail ile Arkadaş Olan Adam,Muğla,Ege Bölgesi


In [6]:
# Haberler verisetini yükle
print("📰 Haberler verisetini yüklüyor...")
dsnews = load_dataset("habanoz/news-tr-1.8M")
dfnews = dsnews['train'].to_pandas()
print(f"✅ Haberler yüklendi. Boyut: {len(dfnews):,} haber")
print(f"📊 Sütunlar: {dfnews.columns.tolist()}")
dfnews.head()


📰 Haberler verisetini yüklüyor...
✅ Haberler yüklendi. Boyut: 1,845,941 haber
📊 Sütunlar: ['text', 'url', 'tokens', 'length']


,text,url,tokens,length
0,10. Büyükelçiler Konferansı\nPutin ve Muhammed...,https://www.aa.com.tr/tr/10-buyukelciler-konfe...,588,1187
1,15 Temmuz şehidi Metin Arslan'ın ablaları üzgü...,https://www.aa.com.tr/tr/15-temmuz-darbe-giris...,840,1678
2,15 Temmuz Şehitler Makamı ve Hafıza 15 Temmuz ...,https://www.aa.com.tr/tr/15-temmuz-darbe-giris...,1408,2853
3,15 Temmuz şehitleri ikiz polislerin babasının ...,https://www.aa.com.tr/tr/15-temmuz-darbe-giris...,1571,3117
4,Ankara\nSakarya\nFetullahçı Terör Örgütü'nün (...,https://www.aa.com.tr/tr/15-temmuz-darbe-giris...,30862,63700


## 2. Yorum Verisetleri Yükle (Kullanıcı Tarafından Doldurulacak)


In [7]:
# Hepsiburada yorumları
print("🛒 Hepsiburada yorumları yükleniyor...")
dshepsiburada = load_dataset("alibayram/hepsiburada_yorumlar")  # Kullanıcı dolduracak
dfhepsiburada = dshepsiburada['train'].to_pandas()
print(f"✅ Hepsiburada yüklendi. Boyut: {len(dfhepsiburada):,} yorum")
print(f"📊 Sütunlar: {dfhepsiburada.columns.tolist()}")
dfhepsiburada.head()


🛒 Hepsiburada yorumları yükleniyor...


✅ Hepsiburada yüklendi. Boyut: 2,657,073 yorum
📊 Sütunlar: ['Puan', 'Baslik', 'Yorum']


,Puan,Baslik,Yorum
0,100,Beş Yıldız,Çocukluğumu hatırlattıbana eskiden hep sabunla...
1,60,güzel,sorunsuz teslimat ürün güzel tebrikler
2,100,Hızlı kargo,Ürünü kullandıktan sonra dayanabilirlik süresi...
3,100,Fiyat performans ürünü,bir çok yer ve mağaza geztik fakat en uygunu b...
4,100,Beş Yıldız,Paranızın karşılığını alacağınız güzel bir ürü...


In [8]:
# Beyazperde yorumları - Kullanıcı tarafından doldurulacak
print("🎬 Beyazperde yorumları yükleniyor...")
dsbeyazperde = load_dataset("alibayram/beyazperde_yorumlar")  # Kullanıcı dolduracak
dfbeyazperde = dsbeyazperde['train'].to_pandas()
print(f"✅ Beyazperde yüklendi. Boyut: {len(dfbeyazperde):,} yorum")
print(f"📊 Sütunlar: {dfbeyazperde.columns.tolist()}")
dfbeyazperde.head()


🎬 Beyazperde yorumları yükleniyor...
✅ Beyazperde yüklendi. Boyut: 192,074 yorum
📊 Sütunlar: ['Puan', 'Yorum']


,Puan,Yorum
0,"4,0",Benim tarzım filmlerden en güzel olanlarından ...
1,"4,0",Çok etkileyici bir filmdi herkese tavsiye eder...
2,"3,0",kitabını okuduktan sonra filmini izlediğim içi...
3,"3,0",kitabını çok beğenmiştm ama filmi izleyince sa...
4,"4,0",Eğer bir dram filmi arıyosanız bu filmi izleme...


In [9]:
# Kitapyurdu yorumları - Kullanıcı tarafından doldurulacak
print("📚 Kitapyurdu yorumları yükleniyor...")
dskitapyurdu = load_dataset("alibayram/kitapyurdu_yorumlar")  # Kullanıcı dolduracak
dfkitapyurdu = dskitapyurdu['train'].to_pandas()
print(f"✅ Kitapyurdu yüklendi. Boyut: {len(dfkitapyurdu):,} yorum")
print(f"📊 Sütunlar: {dfkitapyurdu.columns.tolist()}")
dfkitapyurdu.head()


📚 Kitapyurdu yorumları yükleniyor...
✅ Kitapyurdu yüklendi. Boyut: 404,637 yorum
📊 Sütunlar: ['Puan', 'Yorum']


,Puan,Yorum
0,5,Enine boyuna aşk
1,3,Muhalefetin kendine bir özeleştiri yapması ger...
2,5,Oğuzname Türklerin en eski destanlarından Çıkı...
3,5,İlgi çekici ve sürükleyici bir roman Konusunun...
4,3,antik Mısır hakkında yazılmış olan bir çok rom...


In [10]:
# Yorumbudur yorumları - Kullanıcı tarafından doldurulacak
print("💬 Yorumbudur yorumları yükleniyor...")
dsyorumbudur = load_dataset("alibayram/yorumbudur")  # Kullanıcı dolduracak
dfyorumbudur = dsyorumbudur['train'].to_pandas()
print(f"✅ Yorumbudur yüklendi. Boyut: {len(dfyorumbudur):,} yorum")
print(f"📊 Sütunlar: {dfyorumbudur.columns.tolist()}")
dfyorumbudur.head()


💬 Yorumbudur yorumları yükleniyor...
✅ Yorumbudur yüklendi. Boyut: 2,563,449 yorum
📊 Sütunlar: ['Puan', 'Tarih', 'Baslik', 'Yorum', 'Konum', 'Yoruma verilen artı sayısı', 'Yoruma verilen eksi sayısı']


,Puan,Tarih,Baslik,Yorum,Konum,Yoruma verilen artı sayısı,Yoruma verilen eksi sayısı
0,100,2017-10-02,aşure için aldım,Ben aşure dağıtmak için aldım 3 yıldır kullanı...,SÖĞÜTLÜ - Sakarya,3,0
1,100,2019-01-24,Beş Yıldız,"sağlam paketleme , hızlı kargo , Teşekkürler",None,0,0
2,100,2018-03-28,Başarılı,Yulaf gevreği yemek için aldım gayet kullanışl...,None,0,0
3,100,2018-09-03,Beş Yıldız,Paketleme düzgündü ve oldukça hızlı ulaştı eli...,None,0,0
4,100,2017-10-03,Güzel,Tavsiye ederim,ÇEKMEKÖY - İstanbul,0,0


## 3. DataFrame'lerden Metinleri Çıkar


In [11]:
def extract_texts_from_dataframe(df, text_column=None, max_samples=None, dataset_name=""):
    """DataFrame'den metinleri çıkarır"""
    if df is None:
        print(f"❌ {dataset_name} DataFrame yok, atlanıyor")
        return []
    
    # Metin sütununu otomatik tespit et
    if text_column is None:
        possible_columns = ['text', 'content', 'Yorum', 'review', 'comment', 'article', 'metin', 'title', 'Baslik']
        for col in possible_columns:
            if col in df.columns:
                text_column = col
                break
        
        if text_column is None:
            print(f"❌ {dataset_name} - Metin sütunu bulunamadı. Mevcut sütunlar: {df.columns.tolist()}")
            return []
    
    print(f"📝 {dataset_name} - Metin sütunu: {text_column}")
    
    # Metinleri çıkar
    texts = []
    sample_count = min(len(df), max_samples) if max_samples else len(df)
    
    for i in tqdm(range(sample_count), desc=f"{dataset_name} metinleri çıkarılıyor"):
        try:
            text = df.iloc[i][text_column]
            if isinstance(text, str) and len(text.strip()) > 0:
                texts.append(text.strip())
        except:
            continue
    
    print(f"✅ {dataset_name} - {len(texts)} metin çıkarıldı")
    return texts

# Ağırlık ayarları
NORMAL_WEIGHT = 1      # Normal verisetler için ağırlık
YORUM_WEIGHT = 3       # Yorum verisetleri için ağırlık (yazım hatalarını yakalamak için)

print(f"⚖️ Normal veriset ağırlığı: {NORMAL_WEIGHT}x")
print(f"⚖️ Yorum veriset ağırlığı: {YORUM_WEIGHT}x")


⚖️ Normal veriset ağırlığı: 1x
⚖️ Yorum veriset ağırlığı: 3x


In [12]:
# Normal verisetlerden metinleri çıkar
print("🔄 Normal verisetlerden metinler çıkarılıyor...\n")

# Wikipedia metinleri
wiki_texts = extract_texts_from_dataframe(dfwiki, dataset_name="Wikipedia")

# Masallar metinleri  
masal_texts = extract_texts_from_dataframe(dfmasallar, dataset_name="Masallar")

# Haber metinleri
haber_texts = extract_texts_from_dataframe(dfnews, dataset_name="Haberler")

# Normal metinleri birleştir
normal_texts = wiki_texts + masal_texts + haber_texts

print(f"\n📊 Normal veriset özeti:")
print(f"   Wikipedia: {len(wiki_texts):,} metin")
print(f"   Masallar: {len(masal_texts):,} metin") 
print(f"   Haberler: {len(haber_texts):,} metin")
print(f"   Toplam normal metin: {len(normal_texts):,}")


🔄 Normal verisetlerden metinler çıkarılıyor...

📝 Wikipedia - Metin sütunu: text


Wikipedia metinleri çıkarılıyor:   0%|          | 107/534988 [00:00<08:32, 1043.71it/s]

Wikipedia metinleri çıkarılıyor: 100%|██████████| 534988/534988 [00:45<00:00, 11645.75it/s]


✅ Wikipedia - 534987 metin çıkarıldı
📝 Masallar - Metin sütunu: text


Masallar metinleri çıkarılıyor: 100%|██████████| 1528/1528 [00:02<00:00, 705.62it/s] 


✅ Masallar - 1528 metin çıkarıldı
📝 Haberler - Metin sütunu: text


Haberler metinleri çıkarılıyor: 100%|██████████| 1845941/1845941 [02:14<00:00, 13772.81it/s]

✅ Haberler - 1845941 metin çıkarıldı

📊 Normal veriset özeti:
   Wikipedia: 534,987 metin
   Masallar: 1,528 metin
   Haberler: 1,845,941 metin
   Toplam normal metin: 2,382,456


In [13]:
# Yorum verisetlerinden metinleri çıkar
print("💬 Yorum verisetlerinden metinler çıkarılıyor...\n")

# Hepsiburada yorumları
hepsiburada_texts = extract_texts_from_dataframe(dfhepsiburada,dataset_name="Hepsiburada")

# Beyazperde yorumları
beyazperde_texts = extract_texts_from_dataframe(dfbeyazperde, dataset_name="Beyazperde")

# Kitapyurdu yorumları
kitapyurdu_texts = extract_texts_from_dataframe(dfkitapyurdu, dataset_name="Kitapyurdu")

# Yorumbudur yorumları
yorumbudur_texts = extract_texts_from_dataframe(dfyorumbudur, dataset_name="Yorumbudur")

# Yorum metinlerini birleştir
yorum_texts = hepsiburada_texts + beyazperde_texts + kitapyurdu_texts + yorumbudur_texts

print(f"\n📊 Yorum veriset özeti:")
print(f"   Hepsiburada: {len(hepsiburada_texts):,} yorum")
print(f"   Beyazperde: {len(beyazperde_texts):,} yorum")
print(f"   Kitapyurdu: {len(kitapyurdu_texts):,} yorum")
print(f"   Yorumbudur: {len(yorumbudur_texts):,} yorum")
print(f"   Toplam yorum metni: {len(yorum_texts):,}")


💬 Yorum verisetlerinden metinler çıkarılıyor...

📝 Hepsiburada - Metin sütunu: Yorum


Hepsiburada metinleri çıkarılıyor: 100%|██████████| 2657073/2657073 [00:27<00:00, 95734.13it/s] 


✅ Hepsiburada - 2657036 metin çıkarıldı
📝 Beyazperde - Metin sütunu: Yorum


Beyazperde metinleri çıkarılıyor: 100%|██████████| 192074/192074 [00:02<00:00, 70377.78it/s]


✅ Beyazperde - 192051 metin çıkarıldı
📝 Kitapyurdu - Metin sütunu: Yorum


Kitapyurdu metinleri çıkarılıyor: 100%|██████████| 404637/404637 [00:04<00:00, 84474.34it/s]


✅ Kitapyurdu - 404637 metin çıkarıldı
📝 Yorumbudur - Metin sütunu: Yorum


Yorumbudur metinleri çıkarılıyor: 100%|██████████| 2563449/2563449 [00:28<00:00, 89769.62it/s] 


✅ Yorumbudur - 2563449 metin çıkarıldı

📊 Yorum veriset özeti:
   Hepsiburada: 2,657,036 yorum
   Beyazperde: 192,051 yorum
   Kitapyurdu: 404,637 yorum
   Yorumbudur: 2,563,449 yorum
   Toplam yorum metni: 5,817,173


## 4. Ağırlıklı Veri Hazırla


In [14]:
# Ağırlıklı veri oluştur
weighted_texts = []

# Normal metinleri ekle (1x ağırlık)
print(f"📝 Normal metinler ekleniyor... (ağırlık: {NORMAL_WEIGHT}x)")
for _ in range(NORMAL_WEIGHT):
    weighted_texts.extend(normal_texts)

# Yorum metinlerini ekle (3x ağırlık)
print(f"💬 Yorum metinleri ekleniyor... (ağırlık: {YORUM_WEIGHT}x)")
for _ in range(YORUM_WEIGHT):
    weighted_texts.extend(yorum_texts)

# İstatistikleri hesapla
total_weighted = len(weighted_texts)
normal_ratio = (len(normal_texts) * NORMAL_WEIGHT) / total_weighted if total_weighted > 0 else 0
yorum_ratio = (len(yorum_texts) * YORUM_WEIGHT) / total_weighted if total_weighted > 0 else 0

print(f"✅ Toplam ağırlıklı metin sayısı: {total_weighted:,}")
print(f"📊 Normal metin oranı: {normal_ratio:.2%}")
print(f"📊 Yorum metni oranı: {yorum_ratio:.2%}")

# Metinleri karıştır
random.seed(42)
random.shuffle(weighted_texts)
print("🔀 Metinler karıştırıldı")


📝 Normal metinler ekleniyor... (ağırlık: 1x)
💬 Yorum metinleri ekleniyor... (ağırlık: 3x)
✅ Toplam ağırlıklı metin sayısı: 19,833,975
📊 Normal metin oranı: 12.01%
📊 Yorum metni oranı: 87.99%
🔀 Metinler karıştırıldı


## 5. BPE Tokenizer Eğit


In [ ]:
# BPE konfigürasyonu
BPE_START_ID = 22869
BPE_END_ID = 32767
BPE_VOCAB_SIZE = BPE_END_ID - BPE_START_ID + 1

print(f"🔧 BPE Konfigürasyonu:")
print(f"   Başlangıç ID: {BPE_START_ID}")
print(f"   Bitiş ID: {BPE_END_ID}")
print(f"   Vocab boyutu: {BPE_VOCAB_SIZE}")

# Tokenizer oluştur
tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

# Trainer oluştur - Düzeltilmiş ayarlar
trainer = BpeTrainer(
    vocab_size=BPE_VOCAB_SIZE,
    min_frequency=16,  # Çok düşüktü, 16'ya çıkarıldı
    limit_alphabet=1200,  # Unicode karakterleri sınırla 
    special_tokens=["<unk>", "<pad>", "<s>", "</s>"]
)

print("✅ Tokenizer ve trainer oluşturuldu")


🔧 BPE Konfigürasyonu:
   Başlangıç ID: 22869
   Bitiş ID: 32767
   Vocab boyutu: 9899
✅ Tokenizer ve trainer oluşturuldu


In [16]:
# Eğitim verilerini geçici dosyaya yaz
temp_file = 'temp_training_data.txt'
print(f"📝 Eğitim verisi {temp_file} dosyasına yazılıyor...")

with open(temp_file, 'w', encoding='utf-8') as f:
    for text in tqdm(weighted_texts, desc="Yazılıyor"):
        f.write(text + '\n')

print(f"✅ {len(weighted_texts):,} metin dosyaya yazıldı")
print(f"📊 Dosya boyutu: {os.path.getsize(temp_file) / (1024*1024):.1f} MB")


📝 Eğitim verisi temp_training_data.txt dosyasına yazılıyor...


Yazılıyor: 100%|██████████| 19833975/19833975 [25:38<00:00, 12887.91it/s] 


✅ 19,833,975 metin dosyaya yazıldı
📊 Dosya boyutu: 8005.8 MB


In [17]:
# Tokenizer'ı eğit
print("🚀 BPE tokenizer eğitimi başlıyor...")
print("⏰ Bu işlem birkaç dakika sürebilir...")

tokenizer.train([temp_file], trainer)

print("✅ BPE tokenizer eğitimi tamamlandı!")

# Geçici dosyayı sil
os.remove(temp_file)
print(f"🗑️ Geçici dosya {temp_file} silindi")


🚀 BPE tokenizer eğitimi başlıyor...
⏰ Bu işlem birkaç dakika sürebilir...



✅ BPE tokenizer eğitimi tamamlandı!
🗑️ Geçici dosya temp_training_data.txt silindi


## 6. BPE JSON Dosyasını Oluştur


In [ ]:
# Tokenizer'dan vocab'u çıkar
vocab = tokenizer.get_vocab()
print(f"📊 Toplam vocab boyutu: {len(vocab)}")

# Special tokenları filtrele ve ID'leri ayarla
special_tokens = {"<unk>", "<pad>", "<s>", "</s>"}
bpe_dict = {}
current_id = BPE_START_ID

# Önce special olmayan tokenleri ekle
for token, original_id in sorted(vocab.items(), key=lambda x: x[1]):
    if token not in special_tokens and current_id <= BPE_END_ID:
        bpe_dict[token] = current_id
        current_id += 1

print(f"✅ BPE dictionary oluşturuldu")
print(f"📊 BPE token sayısı: {len(bpe_dict)}")
print(f"🔢 Kullanılan ID aralığı: {BPE_START_ID} - {current_id - 1}")

# Örnek tokenları göster
print("\n📝 Örnek BPE tokenleri:")
sample_tokens = list(bpe_dict.items())[:10]
for token, token_id in sample_tokens:
    print(f"   '{token}': {token_id}")


📊 Toplam vocab boyutu: 10162
✅ BPE dictionary oluşturuldu
📊 BPE token sayısı: 9899
🔢 Kullanılan ID aralığı: 22869 - 32767

📝 Örnek BPE tokenleri:
   '': 22869
   '!': 22870
   '"': 22871
   '#': 22872
   '$': 22873
   '%': 22874
   '&': 22875
   ''': 22876
   '(': 22877
   ')': 22878


In [ ]:
# JSON dosyasını kaydet
output_file = 'bpe_v09.json'
print(f"💾 BPE dictionary {output_file} dosyasına kaydediliyor...")

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(bpe_dict, f, ensure_ascii=False, indent=4, sort_keys=True)

print(f"✅ {output_file} başarıyla oluşturuldu!")
print(f"📊 Dosya boyutu: {os.path.getsize(output_file) / 1024:.1f} KB")


## 7. Sonuç Analizi ve Test


In [ ]:
# İstatistikleri hesapla
token_lengths = [len(token) for token in bpe_dict.keys()]
avg_token_length = np.mean(token_lengths)
max_token_length = max(token_lengths)
min_token_length = min(token_lengths)

print("📈 BPE v09 İstatistikleri:")
print(f"   Toplam token sayısı: {len(bpe_dict):,}")
print(f"   ID aralığı: {BPE_START_ID} - {max(bpe_dict.values())}")
print(f"   Ortalama token uzunluğu: {avg_token_length:.2f} karakter")
print(f"   En uzun token: {max_token_length} karakter")
print(f"   En kısa token: {min_token_length} karakter")

print("\n🎉 BPE v09 oluşturma işlemi tamamlandı!")
print(f"📁 Dosya konumu: {os.path.abspath(output_file)}")


In [ ]:
# Test için birkaç örnek metin tokenizerdan geçir
test_texts = [
    "Bu bir test metnidir.",
    "yazim hatasi olan metin",  # Kasıtlı yazım hatası
    "Merhaba dünya! Nasılsın?",
    "hızlıca gidiyoruz evimize"  # Küçük harf
]

print("🧪 Test Tokenization:")
for text in test_texts:
    encoded = tokenizer.encode(text)
    tokens = encoded.tokens
    print(f"\n📝 Metin: '{text}'")
    print(f"🔤 Tokenler: {tokens}")
    print(f"🔢 Token sayısı: {len(tokens)}")

print("\n✨ Notebook tamamlandı! Veriset isimlerini doldurup çalıştırabilirsin.")
